<a href="https://colab.research.google.com/github/prasanna-venkatesh-m/Analytics-Forecast/blob/main/Analytics_XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

Data cleaning and Perparing

In [31]:
df = pd.read_csv("/content/data.csv")

In [45]:
df['Date'] = pd.to_datetime(df['Date'])

def generate_features(group):
    group['day_of_week'] = group['Date'].dt.dayofweek
    group['day_of_month'] = group['Date'].dt.day
    group['month'] = group['Date'].dt.month

    group['lag_1'] = group['SumofPrincipalOutstanding'].shift(1)
    group['lag_7'] = group['SumofPrincipalOutstanding'].shift(7)
    group['lag_30'] = group['SumofPrincipalOutstanding'].shift(30)
    group['rolling_mean_7'] = group['SumofPrincipalOutstanding'].shift(1).rolling(window=7).mean()

    return group

df_final = df.groupby('BranchID', group_keys=False).apply(generate_features)
df_final = df_final.dropna(subset=['lag_30', 'rolling_mean_7'])

/tmp/ipykernel_1960/3559314197.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_final = df.groupby('BranchID', group_keys=False).apply(generate_features)


Defining the Features and Targets

In [47]:
features = ['BranchID', 'day_of_week', 'day_of_month', 'month',
            'lag_1', 'lag_7', 'lag_30', 'rolling_mean_7']
target = 'SumofPrincipalOutstanding'

df_final['BranchID'] = df_final['BranchID'].astype('category')

split_date = df_final['Date'].max() - pd.Timedelta(days=30)
train = df_final[df_final['Date'] <= split_date]
valid = df_final[df_final['Date'] > split_date]

X_train, y_train = train[features], train[target]
X_valid, y_valid = valid[features], valid[target]